In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from pathlib import Path
 
# Hoặc nhập từ bàn phím:
IMAGE_PATH = Path(input("Nhập đường dẫn ảnh: ").strip())
file_name = str(IMAGE_PATH)

# Đọc dữ liệu
img=cv2.imread(file_name)
if img is None:
  print("Lỗi: không tìm thấy ảnh")

# DÙNG BỘ LỌC
# Làm mờ
edge_aware = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
# Làm rõ
alpha = 2
sharp = cv2.addWeighted(img, 1 + alpha, edge_aware, -alpha, 0)

titles = ["Ảnh gốc", "Ảnh sau khi làm mờ giữ biên", "Ảnh sau khi làm rõ giữ biên"]
images = [img, edge_aware, sharp]
plt.figure(figsize=(20, 6))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    plt.title(titles[i], fontsize=12)
    plt.axis('off')
plt.tight_layout()
plt.show()

# TÍNH TOÁN CHỈ SỐ CHẤT LƯỢNG
def mse(imageA, imageB):
    imageA = cv2.resize(imageA, (imageB.shape[1], imageB.shape[0]))
    err = np.mean((imageA.astype("float") - imageB.astype("float")) ** 2)
    return err

def psnr(imageA, imageB):
    mse_value = mse(imageA, imageB)
    if mse_value == 0:
        return 100
    PIXEL_MAX = 255.0
    return 20 * np.log10(PIXEL_MAX / np.sqrt(mse_value))

def ssim_index(imageA, imageB):
    # ⟵ dùng BGR→GRAY (chuẩn OpenCV) và truyền data_range để tương thích skimage mới
    imageA = cv2.cvtColor(np.uint8(imageA), cv2.COLOR_BGR2GRAY)
    imageB = cv2.cvtColor(np.uint8(imageB), cv2.COLOR_BGR2GRAY)
    s = ssim(imageA, imageB, data_range=255)
    return s

def evaluate(ref, test, name):
    m = mse(ref, test)
    p = psnr(ref, test)
    s = ssim_index(ref, test)
    print(f"{name:40s}  PSNR={p:8.2f} dB | SSIM={s:6.4f}")

print("ĐÁNH GIÁ CHẤT LƯỢNG ẢNH SAU KHI LỌC")
evaluate(img, edge_aware, "Làm mờ giữ biên bằng lọc thông thấp")
evaluate(img, sharp, "Làm rõ giữ biên bằng lọc thông cao")